# TCC F1 - preparação da base

Este notebook acompanha o fluxo atual do projeto em `src/`.

A ordem usada hoje é:

1. `src/extracao_base.py` baixa os dados brutos da Jolpica e do FastF1.
2. `src/juntar_base.py` junta os CSVs brutos em uma base consolidada.
3. `src/tratar_valores_ausentes.py` trata ausências, valores inválidos e cria indicadores auxiliares.
4. `src/one_hot_encoding.py` monta a base numérica para modelagem.

Os scripts que estão em `src/tmp/` não entram neste fluxo do notebook.


## Configuração

A célula abaixo define os caminhos principais e confirma quais scripts fazem parte do fluxo atual.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd()
SRC = ROOT / "src"
DADOS = ROOT / "dados"
PROCESSADOS = DADOS / "processados"
LOGS = DADOS / "logs"

SCRIPTS_FLUXO = [
    SRC / "extracao_base.py",
    SRC / "juntar_base.py",
    SRC / "tratar_valores_ausentes.py",
    SRC / "one_hot_encoding.py",
]

for script in SCRIPTS_FLUXO:
    print(script.relative_to(ROOT), "->", "ok" if script.exists() else "nao encontrado")


In [ ]:
def rodar_script(caminho):
    """executa um script do projeto usando o mesmo python do notebook"""
    caminho = Path(caminho)
    print("Rodando:", caminho.relative_to(ROOT))
    subprocess.run([sys.executable, str(caminho)], cwd=ROOT, check=True)


## Etapa 1 - extração dos dados brutos

Esta etapa baixa os dados da Jolpica e do FastF1 e salva os CSVs em `dados/`.

Como ela pode demorar e depende de APIs externas, a execução fica desligada por padrão. Para rodar de novo, altere `RODAR_EXTRACAO` para `True`.


In [ ]:
RODAR_EXTRACAO = False

if RODAR_EXTRACAO:
    rodar_script(SRC / "extracao_base.py")
else:
    print("Extração não executada nesta rodada.")
    print("Para baixar tudo novamente, defina RODAR_EXTRACAO = True.")


In [ ]:
arquivos_brutos = [
    DADOS / "resultados_2018_2025.csv",
    DADOS / "pitstops_2018_2025.csv",
    DADOS / "circuitos_2018_2025.csv",
    DADOS / "pilotos_2018_2025.csv",
    DADOS / "calendario_circuitos_2018_2025.csv",
    DADOS / "fastf1_qualifying_2018_2025.csv",
    DADOS / "fastf1_laps_2018_2025.csv",
    DADOS / "fastf1_weather_2018_2025.csv",
    DADOS / "circuitos_manual.csv",
]

pd.DataFrame({
    "arquivo": [str(p.relative_to(ROOT)) for p in arquivos_brutos],
    "existe": [p.exists() for p in arquivos_brutos],
})


## Etapa 2 - base consolidada

O script `juntar_base.py` junta resultados, circuitos, pilotos, pit stops, voltas, qualifying e clima.

A saída desta etapa é `dados/processados/base_consolidada_2018_2025.csv`.


In [ ]:
rodar_script(SRC / "juntar_base.py")


In [ ]:
base_consolidada = pd.read_csv(PROCESSADOS / "base_consolidada_2018_2025.csv")

print("Linhas:", len(base_consolidada))
print("Colunas:", len(base_consolidada.columns))
base_consolidada.head()


In [ ]:
# valida a unidade esperada: uma linha por temporada, corrida e piloto
chaves = ["season", "round", "driver_id"]
print("Duplicadas por chave:", base_consolidada.duplicated(subset=chaves).sum())

base_consolidada[["season", "round"]].drop_duplicates().groupby("season").size()


## Etapa 3 - tratamento de valores ausentes e inválidos

O script `tratar_valores_ausentes.py` parte da base consolidada e gera a base tratada.

Ele também salva um log em `dados/logs/log_tratamento_valores_2018_2025.csv` com os preenchimentos e correções aplicadas.


In [ ]:
rodar_script(SRC / "tratar_valores_ausentes.py")


In [ ]:
base_tratada = pd.read_csv(PROCESSADOS / "base_tratada_2018_2025.csv")
log_tratamento = pd.read_csv(LOGS / "log_tratamento_valores_2018_2025.csv")

print("Linhas:", len(base_tratada))
print("Colunas:", len(base_tratada.columns))
print("NaN restantes:", int(base_tratada.isna().sum().sum()))
base_tratada.head()


In [ ]:
if log_tratamento.empty:
    print("Log vazio: nenhuma alteração registrada.")
else:
    display(log_tratamento["tipo_tratamento"].value_counts().rename("quantidade"))
    display(log_tratamento["metodo"].value_counts().rename("quantidade"))


## Etapa 4 - one-hot encoding e base de modelagem

O script `one_hot_encoding.py` separa o alvo `finish_position`, mantém as colunas numéricas de entrada e transforma as colunas categóricas em colunas binárias.

A saída principal é `dados/processados/base_modelagem_2018_2025.csv`.


In [ ]:
rodar_script(SRC / "one_hot_encoding.py")


In [ ]:
base_modelagem = pd.read_csv(PROCESSADOS / "base_modelagem_2018_2025.csv")

with open(PROCESSADOS / "base_modelagem_metadados_2018_2025.json", encoding="utf-8") as arquivo:
    metadados = json.load(arquivo)

print("Linhas:", len(base_modelagem))
print("Colunas:", len(base_modelagem.columns))
print("Alvo:", metadados["coluna_alvo"])
print("Features numéricas:", len(metadados["colunas_numericas"]))
print("Features one-hot:", len(metadados["colunas_one_hot"]))
base_modelagem.head()


In [ ]:
# checagens finais da matriz de modelagem
print("NaN:", int(base_modelagem.isna().sum().sum()))
print("Colunas textuais:", base_modelagem.select_dtypes(include="object").columns.tolist())
print("Colunas duplicadas:", int(base_modelagem.columns.duplicated().sum()))

base_modelagem.dtypes.value_counts()


## Arquivos finais

Resumo dos principais arquivos gerados pelo fluxo atual.


In [ ]:
arquivos_finais = [
    PROCESSADOS / "base_consolidada_2018_2025.csv",
    PROCESSADOS / "base_tratada_2018_2025.csv",
    PROCESSADOS / "base_modelagem_2018_2025.csv",
    PROCESSADOS / "base_modelagem_metadados_2018_2025.json",
    PROCESSADOS / "one_hot_encoder_2018_2025.joblib",
    LOGS / "log_tratamento_valores_2018_2025.csv",
]

resumo_arquivos = []
for arquivo in arquivos_finais:
    resumo_arquivos.append({
        "arquivo": str(arquivo.relative_to(ROOT)),
        "existe": arquivo.exists(),
        "tamanho_kb": round(arquivo.stat().st_size / 1024, 1) if arquivo.exists() else None,
    })

pd.DataFrame(resumo_arquivos)


## Observação sobre `src/tmp/`

A pasta `src/tmp/` contém scripts de experimentação ou etapas antigas. Eles não são chamados por este notebook porque ainda não foram integrados ao fluxo principal em `src/`.
